In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'CMacedonia'

In [4]:
YEAR = 2023
MONTH = 'August'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.46197,40.60446,2023-08-16,κεντρικης μακεδονιας,αλεξανδρειας,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,478.8,38293.0,86.8,46.40326,0.665551,0.289766,-0.574102,-0.289766,0.638476,0.265832,-0.556791,-0.265832,0.085185,0.082991,0.058767,0.082991,25.234545,30.020909,20.448182,11.510466,1.708510,11.778876,0.439805,21.186067,5.575849,24.421964,8.937268,7.689781,8.693039,196.639523,18483.883785,1629.598106,4,120.472193,7.866005,180.191380,0.0,1.400071,31,99,36,99.0,30,99,12,12,1,6,7,2,0,70,0,0
1,22.07722,41.01522,2023-08-16,κεντρικης μακεδονιας,αλμωπιας,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,985.8,24924.0,28.0,46.57952,0.622028,0.182698,-0.540473,-0.182698,0.627293,0.189436,-0.541933,-0.189436,0.056729,0.051588,0.040831,0.051588,21.560926,25.818148,17.303704,9.238694,1.920162,9.573419,-0.371554,15.120878,3.834306,16.704092,6.474975,0.442170,2.184321,318.166087,25701.766035,44.089931,2,194.572797,144.613300,181.767317,0.0,113.550432,31,93,36,94.0,30,93,12,12,3,5,8,2,0,94,0,0
2,22.89198,40.65603,2023-08-16,κεντρικης μακεδονιας,αμπελοκηπων μενεμενης,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,9.8,49674.0,5319.1,46.65786,0.114842,-0.030675,-0.149812,0.030675,0.161744,-0.017994,-0.193685,0.017994,0.069664,0.029342,0.062271,0.029342,29.190000,35.210000,23.170000,12.381429,3.596667,11.870000,2.525385,18.760769,5.121429,23.898000,11.870000,0.000000,0.000000,416.613286,1013.723139,1259.926671,2,166.963790,4.655965,180.457193,0.0,24.343007,32,97,9,99.0,30,97,13,13,10,8,9,2,0,150,0,0
3,23.95900,40.91196,2023-08-16,κεντρικης μακεδονιας,αμφιπολης,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,411.8,7169.0,22.3,47.41120,0.416450,0.072981,-0.412194,-0.072981,0.409613,0.054098,-0.412106,-0.054098,0.047239,0.046840,0.024948,0.046840,28.570000,35.922000,21.218000,10.655161,3.113550,9.895287,0.416503,16.682835,4.861432,20.539981,7.233860,0.000000,0.527649,127.932595,14413.246350,196.335052,5,220.810138,250.710581,185.823044,0.0,25.464784,31,88,30,88.0,30,88,10,10,1,6,6,2,0,95,0,0
4,23.69747,40.49593,2023-08-16,κεντρικης μακεδονιας,αριστοτελη,16,8,33,2023,-0.101168,-0.994869,-0.866025,-0.5,-0.696551,-0.717507,747.0,16994.0,24.5,46.92004,0.662182,0.270805,-0.571899,-0.270805,0.660042,0.263737,-0.569933,-0.263737,0.057341,0.052713,0.037928,0.052713,24.242778,28.485000,20.000556,10.968774,3.770478,9.453781,2.661346,15.362634,4.844974,17.259672,7.381085,0.000000,0.000944,341.157482,11468.297401,1749.385768,22,273.587292,768.043224,181.562024,0.0,1.033661,14,99,10,99.0,10,99,4,4,6,4,4,2,0,80,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.72632,40.61725,δελτα,16,8,2023,0.985249
1,22.95369,40.62334,θεσσαλονικης,16,8,2023,0.984257
2,22.36451,40.79600,πελλας,16,8,2023,0.983069
3,22.46197,40.60446,αλεξανδρειας,16,8,2023,0.979208
4,22.90326,40.67609,κορδελιου ευοσμου,16,8,2023,0.977118
5,22.37828,40.28551,κατερινης,16,8,2023,0.972779
6,23.08410,40.49006,θερμης,16,8,2023,0.968325
7,22.91743,40.42897,θερμαϊκου,16,8,2023,0.966276
8,22.90397,41.04735,κιλκις,16,8,2023,0.965397
9,23.27201,41.13057,ηρακλειας,16,8,2023,0.964346


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0648252687608289, 0.3787123377107355, 0.6803072708219687, 0.838221147727126, 0.935474790938512, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.72632,40.61725,δελτα,16,8,2023,0.985249,5
1,22.95369,40.62334,θεσσαλονικης,16,8,2023,0.984257,5
2,22.36451,40.79600,πελλας,16,8,2023,0.983069,5
3,22.46197,40.60446,αλεξανδρειας,16,8,2023,0.979208,5
4,22.90326,40.67609,κορδελιου ευοσμου,16,8,2023,0.977118,5
5,22.37828,40.28551,κατερινης,16,8,2023,0.972779,5
6,23.08410,40.49006,θερμης,16,8,2023,0.968325,5
7,22.91743,40.42897,θερμαϊκου,16,8,2023,0.966276,5
8,22.90397,41.04735,κιλκις,16,8,2023,0.965397,5
9,23.27201,41.13057,ηρακλειας,16,8,2023,0.964346,5


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results